# EA1 - Actividad 1.3: DataFrames en Spark

## Objetivos
- Dominar operaciones de DataFrame: select, filter, groupBy, join
- Utilizar funciones de agregacion y funciones de columna
- Trabajar con funciones de fecha
- Crear columnas derivadas con `withColumn`, `when/otherwise`

## Conceptos Clave

### DataFrames vs RDDs

| Aspecto | RDD | DataFrame |
|---------|-----|-----------|
| **Esquema** | Sin esquema (datos no estructurados) | Con esquema (columnas tipadas) |
| **Optimizacion** | Sin optimizacion automatica | Optimizador **Catalyst** + motor **Tungsten** |
| **API** | Funcional (map, filter, reduce) | Declarativa (select, filter, groupBy) - similar a SQL |
| **Rendimiento** | Menor (sin optimizaciones) | Mayor (plan de ejecucion optimizado) |
| **Uso recomendado** | Datos no estructurados, control fino | Datos estructurados/semi-estructurados |

**Recomendacion:** Usar DataFrames siempre que sea posible. Los RDDs son utiles cuando necesitas control a bajo nivel o trabajas con datos no estructurados.

In [ ]:
# Setup: Crear SparkSession e importar funciones
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("03_dataframes_intro") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

## Cargar los datos

Trabajaremos con dos datasets:
- **sales.csv:** Datos de ventas semanales por tienda y departamento
- **stores.csv:** Informacion de cada tienda (tipo y tamano)

In [ ]:
# Leer los datasets
df_sales = spark.read.csv("/home/jovyan/datos/sales.csv", header=True, inferSchema=True)
df_stores = spark.read.csv("/home/jovyan/datos/stores.csv", header=True, inferSchema=True)

In [ ]:
# Explorar df_sales
print("=== SALES ===")
df_sales.show(5)
df_sales.printSchema()
print(f"Filas: {df_sales.count()}, Columnas: {len(df_sales.columns)}")

In [ ]:
# Explorar df_stores
print("=== STORES ===")
df_stores.show(5)
df_stores.printSchema()
print(f"Filas: {df_stores.count()}, Columnas: {len(df_stores.columns)}")

## `select()` y `withColumn()`

- **`select()`**: Elige columnas especificas (como SELECT en SQL)
- **`withColumn()`**: Agrega o reemplaza una columna

In [ ]:
# select: elegir columnas especificas
df_sales.select("Store", "Weekly_Sales").show(5)

# withColumn: crear una nueva columna
# Ejemplo: convertir ventas a una columna explicita en USD
df_con_usd = df_sales.withColumn("Sales_USD", F.col("Weekly_Sales") * 1.0)
df_con_usd.select("Store", "Weekly_Sales", "Sales_USD").show(5)

## `filter()` con condiciones multiples

Se pueden combinar condiciones usando `&` (AND) y `|` (OR). Cada condicion debe ir entre parentesis.

In [ ]:
# Filtrar: ventas mayores a 50000 en dias festivos
df_filtrado = df_sales.filter(
    (F.col("Weekly_Sales") > 50000) & (F.col("IsHoliday") == True)
)

print(f"Registros con ventas > 50K en festivos: {df_filtrado.count()}")
df_filtrado.show(5)

## `groupBy()` + `agg()`

Agrupar datos y aplicar funciones de agregacion: `sum`, `avg`, `count`, `min`, `max`, etc.

In [ ]:
# Agrupar por tienda y calcular total y promedio de ventas
df_por_tienda = df_sales.groupBy("Store").agg(
    F.sum("Weekly_Sales").alias("total_ventas"),
    F.avg("Weekly_Sales").alias("promedio_ventas"),
    F.count("Weekly_Sales").alias("num_registros")
)

df_por_tienda.orderBy("total_ventas", ascending=False).show(10)

## `join()` - Unir DataFrames

El join permite combinar dos DataFrames basandose en una columna en comun, similar a JOIN en SQL.

In [ ]:
# Join: unir ventas con informacion de tiendas
df_joined = df_sales.join(df_stores, "Store")

print(f"Columnas despues del join: {df_joined.columns}")
df_joined.show(5)

## Funciones de Fecha

Spark incluye funciones para trabajar con columnas de fecha: parsear, extraer componentes, calcular diferencias, etc.

In [ ]:
# Convertir la columna Date a tipo fecha (si no lo es ya)
df_con_fecha = df_sales.withColumn("Fecha", F.to_date(F.col("Date"), "yyyy-MM-dd"))

# Extraer componentes de la fecha
df_con_fecha = df_con_fecha \
    .withColumn("Anio", F.year("Fecha")) \
    .withColumn("Mes", F.month("Fecha")) \
    .withColumn("Dia_Semana", F.dayofweek("Fecha"))

df_con_fecha.select("Store", "Date", "Fecha", "Anio", "Mes", "Dia_Semana").show(5)

## `when()` / `otherwise()` - Columnas condicionales

Permite crear columnas basadas en condiciones, similar a CASE WHEN en SQL.

In [ ]:
# Crear categoria de ventas basada en el monto
df_categorizado = df_sales.withColumn(
    "categoria",
    F.when(F.col("Weekly_Sales") > 50000, "Alta")
     .when(F.col("Weekly_Sales") > 20000, "Media")
     .otherwise("Baja")
)

df_categorizado.select("Store", "Weekly_Sales", "categoria").show(10)

# Contar cuantos registros hay en cada categoria
df_categorizado.groupBy("categoria").count().show()

---
## Ejercicios

Ahora es tu turno de practicar con DataFrames.

In [ ]:
# =============================================================
# EJERCICIO 1: Ventas totales por tipo de tienda
# =============================================================
# TODO: Calcula las ventas totales por tipo de tienda.
#   1. Haz un join entre df_sales y df_stores usando la columna "Store"
#   2. Agrupa por la columna "Type" (tipo de tienda: A, B, C)
#   3. Calcula la suma de "Weekly_Sales" con alias "total_ventas"
#   4. Ordena de mayor a menor y muestra el resultado
#
# Pista:
#   df_sales.join(df_stores, "Store")
#           .groupBy("Type")
#           .agg(F.sum("Weekly_Sales").alias("total_ventas"))
#           .orderBy("total_ventas", ascending=False)
#           .show()

# Escribe tu codigo aqui:


In [ ]:
# =============================================================
# EJERCICIO 2: Tienda con mayor venta promedio semanal
# =============================================================
# TODO: Encuentra la tienda con la mayor venta promedio semanal.
#   1. Agrupa df_sales por "Store"
#   2. Calcula el promedio de "Weekly_Sales" con alias "promedio_semanal"
#   3. Ordena de mayor a menor por "promedio_semanal"
#   4. Muestra solo la primera fila (la tienda con mayor promedio)
#
# Pista: .orderBy("promedio_semanal", ascending=False).show(1)

# Escribe tu codigo aqui:


In [ ]:
# =============================================================
# EJERCICIO 3: Crear columna "temporada" basada en el mes
# =============================================================
# TODO: Agrega una columna "temporada" al DataFrame basada en el mes.
#   Usa las estaciones del hemisferio sur (Chile):
#     - Diciembre, Enero, Febrero -> "Verano"
#     - Marzo, Abril, Mayo -> "Otono"
#     - Junio, Julio, Agosto -> "Invierno"
#     - Septiembre, Octubre, Noviembre -> "Primavera"
#
# Pasos:
#   1. Primero extrae el mes de la columna "Date":
#      df_temp = df_sales.withColumn("Mes", F.month(F.to_date("Date", "yyyy-MM-dd")))
#   2. Luego usa F.when() encadenado para crear la columna "temporada"
#      basandote en el valor de "Mes"
#   3. Muestra algunos registros y haz un groupBy("temporada").count()
#
# Pista: F.col("Mes").isin(12, 1, 2) para verificar si el mes esta en una lista

# Escribe tu codigo aqui:


---
## Resumen

En esta actividad aprendimos:

1. **DataFrames vs RDDs:** Los DataFrames tienen esquema y se benefician del optimizador Catalyst
2. **select() y withColumn():** Seleccionar y crear columnas
3. **filter():** Filtrar filas con condiciones simples y compuestas
4. **groupBy() + agg():** Agrupar datos y calcular agregaciones (sum, avg, count, etc.)
5. **join():** Unir DataFrames por columnas en comun
6. **Funciones de fecha:** to_date, year, month, dayofweek
7. **when/otherwise:** Crear columnas condicionales

---
## Desafio Extra (Opcional)

**Crecimiento porcentual de ventas mes a mes por tienda**

Usa **Window Functions** para calcular el crecimiento porcentual de ventas de un mes al siguiente para cada tienda.

Pistas:
- Primero agrega ventas por tienda y mes
- Usa `from pyspark.sql.window import Window`
- Define una ventana: `Window.partitionBy("Store").orderBy("Mes")`
- Usa `F.lag("total_ventas", 1)` para obtener el valor del mes anterior
- Calcula: `((actual - anterior) / anterior) * 100`

In [ ]:
# =============================================================
# DESAFIO: Crecimiento porcentual mes a mes por tienda
# =============================================================
# TODO: Calcula el crecimiento porcentual de ventas mes a mes
#   para cada tienda usando Window Functions.
#
# Pasos sugeridos:
#   1. Extraer mes y agrupar por Store y Mes, sumando Weekly_Sales
#   2. Crear una Window particionada por Store y ordenada por Mes
#   3. Usar F.lag() para obtener ventas del mes anterior
#   4. Calcular el porcentaje de crecimiento
#   5. Mostrar resultados para 2-3 tiendas

from pyspark.sql.window import Window

# Escribe tu codigo aqui:


In [ ]:
# Detener la SparkSession al finalizar
spark.stop()
print("SparkSession detenida correctamente.")